# Seed-disagreement filtering of synthetic road imagery — cross-validated experiments

Runs the real-only evaluation protocol (`experiments/` package) on Colab.
Results are written under `RESULTS_DIR` on Google Drive so the grid can be
resumed across sessions: the driver skips every cell that already has a
`result.json`.

Steps: mount Drive → clone/update the repo → point `data/` at the dataset →
run the grid → aggregate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL    = 'https://github.com/hz6yc3/attention-enhanced-unet-segmentation.git'  # <-- edit
DATA_DIR    = '/content/drive/MyDrive/road_segmentation/data'      # contains training/ and test_set_images/
RESULTS_DIR = '/content/drive/MyDrive/road_segmentation/results'   # persists across sessions


In [ ]:
import os
if not os.path.exists('/content/repo'):
    !git clone $REPO_URL /content/repo
%cd /content/repo
!git pull
!pip -q install albumentations scipy pandas
!nvidia-smi -L


In [ ]:
# Link the dataset into the repo layout and create the frozen splits
!rm -rf data && ln -s $DATA_DIR data
!python -m utils.splits --data data


## One-command run (keeps going while you do other things, as long as the session is alive)

Launches the whole plan with `nohup` so the cell returns immediately. Colab still ends the
session on inactivity or its time limit, so keep the tab open or use Colab Pro+ background
execution. Results and the log are on Drive, so re-running this cell after a disconnect resumes.

In [ ]:
!nohup bash scripts/run_budget.sh $RESULTS_DIR 1200 core,all > /dev/null 2>&1 &
print('started; progress in', RESULTS_DIR + '/run_budget.log')


In [ ]:
# Check progress any time
!tail -5 $RESULTS_DIR/run_budget.log; ls $RESULTS_DIR/runs 2>/dev/null | wc -l


## Budgeted plan (about 3 GPU-hours on a T4)

Priority order. Each stage is resumable and adds to the same results folder.

1. **Core claim** — Attention U-Net, 5 folds x 3 seeds, conditions `real`, `random`, `filtered` (k = 250), 1200 steps per run: 45 runs + 5 scoring passes.
2. **If time remains** — add `all` (real + every synthetic image), 15 runs.
3. **Later / optional** — baseline U-Net, `antifiltered` control, dose-response over k.

Time one run first: the log line `done in X min` tells you the per-run cost; multiply by the run count before launching a stage.

In [ ]:
# Timing check: one real-only run (about 2-4 min on a T4). Its result is reused by stage 1.
!python -m experiments.train_run --arch attention --fold 0 --seed 42 --condition real \
    --max-steps 1200 --eval-every 100 --out-dir $RESULTS_DIR


In [ ]:
# Stage 1 - core claim (45 runs + scoring). Re-run after a timeout; finished cells are skipped.
!python -m experiments.run_cv --archs attention --folds 0 1 2 3 4 --seeds 42 123 456 \
    --conditions random filtered --k 250 --max-steps 1200 --eval-every 100 \
    --out-dir $RESULTS_DIR --quiet


In [ ]:
# Stage 2 - if time remains: real + ALL synthetic images (15 runs)
!python -m experiments.run_cv --archs attention --folds 0 1 2 3 4 --seeds 42 123 456 \
    --conditions all --max-steps 1200 --eval-every 100 --out-dir $RESULTS_DIR --quiet


In [ ]:
# Stage 3 - optional extras (baseline U-Net, antifiltered control, dose-response)
# !python -m experiments.run_cv --archs baseline --conditions random filtered --k 250 --max-steps 1200 --out-dir $RESULTS_DIR --quiet
# !python -m experiments.run_cv --archs attention --conditions antifiltered --k 250 --max-steps 1200 --out-dir $RESULTS_DIR --quiet
# !python -m experiments.run_cv --archs attention --conditions random filtered --k 100 500 --max-steps 1200 --out-dir $RESULTS_DIR --quiet


In [ ]:
!python -m experiments.aggregate --results $RESULTS_DIR
